# Handwritten Digit Classification: kNN vs Feed-Forward Neural Networks

**Problem Statement:** Handwritten digit recognition is a classic supervised learning task. In this project, we compare two different approaches on the same data: a from-scratch k-nearest neighbors classifier and a from-scratch feed-forward neural network.

The following code is provided to help you get started. The default settings use a small built-in digits dataset so the notebook can run quickly on CPU. For a larger project, you can extend the same ideas to MNIST or another image classification dataset.


## 1. Install and Import Dependencies

In [ ]:
# !pip install scikit-learn matplotlib numpy tqdm

import math
import random
import time
from collections import Counter
from typing import Dict, List, Sequence

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm


## 2. Load and Explore the Digits Dataset

The scikit-learn digits dataset contains 8 by 8 grayscale images of handwritten digits. Each image is flattened into 64 input features. Pixel values are normalized to the range 0 to 1.


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

digits = load_digits()
X = digits.data.astype(np.float64) / 16.0
y = digits.target.astype(int)

X_train_full, X_temp, y_train_full, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val_full, X_test_full, y_val_full, y_test_full = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

# Keep the default FFNN run quick. Increase these values for a stronger experiment.
TRAIN_LIMIT = 200
VAL_LIMIT = 80
TEST_LIMIT = 80

X_train = X_train_full[:TRAIN_LIMIT]
y_train = y_train_full[:TRAIN_LIMIT]
X_val = X_val_full[:VAL_LIMIT]
y_val = y_val_full[:VAL_LIMIT]
X_test = X_test_full[:TEST_LIMIT]
y_test = y_test_full[:TEST_LIMIT]

print(f"Train: {X_train.shape}, Validation: {X_val.shape}, Test: {X_test.shape}")
print(f"Classes: {[int(label) for label in sorted(set(y))]}")

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.ravel(), X_train[:10], y_train[:10]):
    ax.imshow(image.reshape(8, 8), cmap='gray_r')
    ax.set_title(f"Label: {label}")
    ax.axis('off')
plt.tight_layout()
plt.show()


## 3. From-Scratch kNN Baseline

k-nearest neighbors classifies a new point by finding the most similar training examples. It does not learn weights during training; most of the work happens at prediction time.


In [ ]:
def euclidean_distances(X_reference: np.ndarray, x_query: np.ndarray) -> np.ndarray:
    """Return the Euclidean distance from x_query to every row in X_reference."""
    differences = X_reference - x_query
    return np.sqrt(np.sum(differences * differences, axis=1))


def knn_predict_one(
    X_reference: np.ndarray,
    y_reference: np.ndarray,
    x_query: np.ndarray,
    k: int = 3,
) -> int:
    """Predict one label using majority vote among the k nearest examples."""
    distances = euclidean_distances(X_reference, x_query)
    nearest_indices = np.argsort(distances)[:k]
    nearest_labels = y_reference[nearest_indices]
    label_counts = Counter(nearest_labels)

    # Tie-break by choosing the smaller digit label for deterministic results.
    return min(label_counts, key=lambda label: (-label_counts[label], label))


def knn_predict(
    X_reference: np.ndarray,
    y_reference: np.ndarray,
    X_query: np.ndarray,
    k: int = 3,
) -> np.ndarray:
    """Predict labels for every row in X_query."""
    predictions = [knn_predict_one(X_reference, y_reference, row, k=k) for row in X_query]
    return np.array(predictions, dtype=int)


def accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Return classification accuracy as a percentage."""
    return float(np.mean(y_true == y_pred) * 100.0)


In [ ]:
knn_results = []
for k in [1, 3, 5, 7]:
    start = time.perf_counter()
    val_predictions = knn_predict(X_train, y_train, X_val, k=k)
    elapsed = time.perf_counter() - start
    val_acc = accuracy(y_val, val_predictions)
    knn_results.append({'k': k, 'val_accuracy': val_acc, 'seconds': elapsed})
    print(f"k={k}: validation accuracy={val_acc:.2f}% ({elapsed:.2f}s)")

best_knn = max(knn_results, key=lambda row: row['val_accuracy'])
best_k = best_knn['k']

start = time.perf_counter()
knn_test_predictions = knn_predict(X_train, y_train, X_test, k=best_k)
knn_test_time = time.perf_counter() - start
knn_test_accuracy = accuracy(y_test, knn_test_predictions)

print(f"\nBest k: {best_k}")
print(f"kNN test accuracy: {knn_test_accuracy:.2f}% ({knn_test_time:.2f}s)")


## 4. From-Scratch FFNN Baseline

A feed-forward neural network learns weights by adjusting them with gradient descent. The code below builds up a small computation graph: each scalar value records its data, its gradient, and how it was produced, so we can backpropagate by hand without PyTorch or TensorFlow.


In [ ]:
def create_value(data: float, _children: Sequence[Dict] = (), _op: str = '') -> Dict:
    """Create one scalar Value node for the computation graph."""
    return {
        'data': float(data),
        'grad': 0.0,
        '_prev': list(_children),
        '_op': _op,
        '_backward': lambda: None,
    }


def add(a: Dict, b: Dict) -> Dict:
    out = create_value(a['data'] + b['data'], (a, b), '+')

    def _backward():
        a['grad'] += out['grad']
        b['grad'] += out['grad']

    out['_backward'] = _backward
    return out


def multiply(a: Dict, b: Dict) -> Dict:
    out = create_value(a['data'] * b['data'], (a, b), '*')

    def _backward():
        a['grad'] += b['data'] * out['grad']
        b['grad'] += a['data'] * out['grad']

    out['_backward'] = _backward
    return out


def power(a: Dict, exponent: float) -> Dict:
    out = create_value(a['data'] ** exponent, (a,), f'**{exponent}')

    def _backward():
        a['grad'] += exponent * (a['data'] ** (exponent - 1)) * out['grad']

    out['_backward'] = _backward
    return out


def relu(a: Dict) -> Dict:
    out = create_value(max(0.0, a['data']), (a,), 'relu')

    def _backward():
        a['grad'] += (1.0 if a['data'] > 0 else 0.0) * out['grad']

    out['_backward'] = _backward
    return out


def exp_op(a: Dict) -> Dict:
    clipped = max(min(a['data'], 50.0), -50.0)
    out = create_value(math.exp(clipped), (a,), 'exp')

    def _backward():
        a['grad'] += out['data'] * out['grad']

    out['_backward'] = _backward
    return out


def log_op(a: Dict) -> Dict:
    safe_value = max(a['data'], 1e-12)
    out = create_value(math.log(safe_value), (a,), 'log')

    def _backward():
        a['grad'] += (1.0 / safe_value) * out['grad']

    out['_backward'] = _backward
    return out


In [ ]:
def softmax(logits: List[Dict]) -> List[Dict]:
    """Convert raw class scores into probabilities."""
    max_logit = max(node['data'] for node in logits)
    shifted = [add(node, create_value(-max_logit)) for node in logits]
    exp_values = [exp_op(node) for node in shifted]

    total = create_value(0.0)
    for value in exp_values:
        total = add(total, value)

    inv_total = power(total, -1.0)
    return [multiply(value, inv_total) for value in exp_values]


def initialize_model(
    input_size: int,
    layer_sizes: Sequence[int],
    activations: Sequence[str],
    seed: int = 0,
) -> Dict:
    """Create an MLP stored as dictionaries and lists."""
    rng = random.Random(seed)
    layers = []
    previous_size = input_size

    for layer_size, activation in zip(layer_sizes, activations):
        scale = math.sqrt(2.0 / previous_size) if activation == 'relu' else math.sqrt(1.0 / previous_size)
        neurons = []
        for _ in range(layer_size):
            weights = [create_value(rng.gauss(0.0, scale)) for _ in range(previous_size)]
            bias = create_value(0.0)
            neurons.append({'weights': weights, 'bias': bias})
        layers.append({'neurons': neurons, 'activation': activation})
        previous_size = layer_size

    return {'layers': layers}


def forward(model: Dict, x: Sequence[float]) -> List[Dict]:
    """Run one example through the MLP."""
    activations = [create_value(value) for value in x]

    for layer in model['layers']:
        linear_outputs = []
        for neuron in layer['neurons']:
            total = neuron['bias']
            for weight, activation in zip(neuron['weights'], activations):
                total = add(total, multiply(weight, activation))
            linear_outputs.append(total)

        if layer['activation'] == 'relu':
            activations = [relu(value) for value in linear_outputs]
        elif layer['activation'] == 'softmax':
            activations = softmax(linear_outputs)
        else:
            activations = linear_outputs

    return activations


def get_parameters(model: Dict) -> List[Dict]:
    """Return every trainable Value node in the model."""
    parameters = []
    for layer in model['layers']:
        for neuron in layer['neurons']:
            parameters.extend(neuron['weights'])
            parameters.append(neuron['bias'])
    return parameters


def cross_entropy_loss(probabilities: List[Dict], target_label: int) -> Dict:
    """Return -log(probability assigned to the correct class)."""
    return multiply(create_value(-1.0), log_op(probabilities[target_label]))


def backward(loss: Dict) -> None:
    """Backpropagate from loss through the computation graph."""
    topo = []
    visited = set()

    def build_topo(node: Dict):
        node_id = id(node)
        if node_id in visited:
            return
        visited.add(node_id)
        for parent in node.get('_prev', []):
            build_topo(parent)
        topo.append(node)

    build_topo(loss)
    loss['grad'] = 1.0
    for node in reversed(topo):
        node['_backward']()


In [ ]:
def ffnn_predict_one(model: Dict, x: Sequence[float]) -> int:
    outputs = forward(model, x)
    return int(np.argmax([node['data'] for node in outputs]))


def ffnn_accuracy(model: Dict, X_eval: np.ndarray, y_eval: np.ndarray) -> float:
    predictions = [ffnn_predict_one(model, row) for row in X_eval]
    return accuracy(y_eval, np.array(predictions, dtype=int))


def average_loss(model: Dict, X_eval: np.ndarray, y_eval: np.ndarray) -> float:
    total_loss = 0.0
    for row, label in zip(X_eval, y_eval):
        outputs = forward(model, row)
        total_loss += cross_entropy_loss(outputs, int(label))['data']
    return total_loss / len(X_eval)


def train_ffnn(
    model: Dict,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    epochs: int = 5,
    learning_rate: float = 0.03,
) -> Dict[str, List[float]]:
    """Train the FFNN with stochastic gradient descent."""
    parameters = get_parameters(model)
    history = {'train_loss': [], 'train_accuracy': [], 'val_loss': [], 'val_accuracy': []}

    for epoch in range(epochs):
        indices = np.random.permutation(len(X_train))
        epoch_loss = 0.0
        start = time.perf_counter()

        for idx in tqdm(indices, desc=f"Epoch {epoch + 1}/{epochs}", leave=False):
            for parameter in parameters:
                parameter['grad'] = 0.0

            outputs = forward(model, X_train[idx])
            loss = cross_entropy_loss(outputs, int(y_train[idx]))
            backward(loss)

            for parameter in parameters:
                parameter['data'] -= learning_rate * parameter['grad']

            epoch_loss += loss['data']

        train_loss = epoch_loss / len(X_train)
        train_eval_count = min(100, len(X_train))
        train_acc = ffnn_accuracy(model, X_train[:train_eval_count], y_train[:train_eval_count])
        val_loss = average_loss(model, X_val, y_val)
        val_acc = ffnn_accuracy(model, X_val, y_val)
        elapsed = time.perf_counter() - start

        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)

        print(
            f"Epoch {epoch + 1}: train loss={train_loss:.3f}, "
            f"train acc={train_acc:.1f}%, val acc={val_acc:.1f}% ({elapsed:.1f}s)"
        )

    return history


In [ ]:
ffnn_model = initialize_model(
    input_size=X_train.shape[1],
    layer_sizes=[16, 10],
    activations=['relu', 'softmax'],
    seed=SEED,
)
print(f"FFNN parameters: {len(get_parameters(ffnn_model))}")

ffnn_start = time.perf_counter()
ffnn_history = train_ffnn(
    ffnn_model,
    X_train,
    y_train,
    X_val,
    y_val,
    epochs=4,
    learning_rate=0.03,
)
ffnn_train_time = time.perf_counter() - ffnn_start
ffnn_test_accuracy = ffnn_accuracy(ffnn_model, X_test, y_test)

print(f"\nFFNN test accuracy: {ffnn_test_accuracy:.2f}%")
print(f"FFNN training time: {ffnn_train_time:.2f}s")


In [ ]:
def plot_ffnn_history(history: Dict[str, List[float]]) -> None:
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history['train_loss'], marker='o', label='Train')
    axes[0].plot(epochs, history['val_loss'], marker='o', label='Validation')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history['train_accuracy'], marker='o', label='Train')
    axes[1].plot(epochs, history['val_accuracy'], marker='o', label='Validation')
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


plot_ffnn_history(ffnn_history)


## 5. Compare kNN and FFNN Results

Both models saw the same training, validation, and test split. kNN stores the training data and compares each test image to it. The FFNN spends time learning weights, then predicts with a fixed set of parameters.


In [ ]:
comparison = [
    {
        'model': f'kNN (k={best_k})',
        'validation_accuracy': best_knn['val_accuracy'],
        'test_accuracy': knn_test_accuracy,
        'time_seconds': knn_test_time,
        'notes': 'Prediction-time distance search',
    },
    {
        'model': 'FFNN (64 -> 16 -> 10)',
        'validation_accuracy': ffnn_history['val_accuracy'][-1],
        'test_accuracy': ffnn_test_accuracy,
        'time_seconds': ffnn_train_time,
        'notes': 'Training-time weight learning',
    },
]

print(f"{'Model':<24} {'Val Acc':>10} {'Test Acc':>10} {'Time (s)':>10}  Notes")
print('-' * 78)
for row in comparison:
    print(
        f"{row['model']:<24} "
        f"{row['validation_accuracy']:>9.2f}% "
        f"{row['test_accuracy']:>9.2f}% "
        f"{row['time_seconds']:>10.2f}  "
        f"{row['notes']}"
    )


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, true_label in zip(axes.ravel(), X_test[:10], y_test[:10]):
    knn_label = knn_predict_one(X_train, y_train, image, k=best_k)
    ffnn_label = ffnn_predict_one(ffnn_model, image)
    ax.imshow(image.reshape(8, 8), cmap='gray_r')
    ax.set_title(f"True {true_label}\nkNN {knn_label}, FFNN {ffnn_label}")
    ax.axis('off')
plt.tight_layout()
plt.show()


## 6. Experiment Ideas

Use this notebook as a starting point for a project. Natural directions include:

- Try larger train, validation, and test limits. How do the two models scale?
- Try other values of `k` and different distance functions for kNN.
- Change the FFNN hidden layer size, learning rate, number of epochs, or activation function.
- Compare models on specific digits. Which classes are confused most often?
- Extend the notebook to MNIST, Fashion-MNIST, or another image classification dataset.
- Vectorize the FFNN with NumPy so it can train on more data faster.


In [ ]:
# TODO: Your experiments go here.
# For example: increase TRAIN_LIMIT, tweak the FFNN architecture, or rerun with a different random seed.
